# Label Generation: 1-Day, 2-Day, and 7-Day Prediction

## Prediction Frameworks

### 1-Day Rolling Prediction
| Day | Role |
|---|---|
| t | Input horizon (current day data) |
| t+1 | Gap window (excluded — prevents leakage) |
| t+2 | Prediction target → **label = 1** |

### 2-Day Rolling Prediction
| Day | Role |
|---|---|
| t | Input horizon |
| t+1 | Gap window (excluded) |
| t+2, t+3 | Prediction window (2 days) → **label = 1** |

### 7-Day Non-Rolling Prediction
- Input: **Days 1–5** of each patient visit
- **label = 1** if Day 7 of that visit is the patient's death day
- **label = 0** otherwise (survivor, not enough days, or death not on Day 7)

---
Each output includes:
- `Day`: day count from beginning of the visit
- `days_until_death`: calendar days remaining until death from that day

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
base_dir = "../../Data"

# CRRT pressure streams — output of denoising pipeline
# Columns: PID, Visit_no, Day, Time, ACCESS_PRESSURE, FILTER_PRESSURE, EFFLUENT_PRESSURE, RETURN_PRESSURE
crrt = pd.read_csv(
    os.path.join(base_dir, r"Processed\Machine\pressure\Denoised\CRRT_pressure_wo_peak.csv"),
    index_col=False
)

# Mortality labels — Columns: PID, Mortality, Death_Date
mortality = pd.read_csv(
    os.path.join(base_dir, r"Intermediate\mortality_label.csv"),
    index_col=False
)

print(f"CRRT shape      : {crrt.shape}")
print(f"Mortality shape : {mortality.shape}")
print(f"\nUnique PIDs in CRRT     : {crrt['PID'].nunique()}")
print(f"Unique PIDs in Mortality: {mortality['PID'].nunique()}")

In [ ]:
# Parse dates
crrt['Date'] = pd.to_datetime(crrt['Time']).dt.date
mortality['Death_Date'] = pd.to_datetime(mortality['Death_Date'], errors='coerce').dt.date

# Merge mortality info into CRRT
crrt = crrt.merge(mortality[['PID', 'Mortality', 'Death_Date']], on='PID', how='left')

# Collapse to one row per (PID, Visit_no, Day)
day_df = crrt.groupby(['PID', 'Visit_no', 'Day'], as_index=False).agg(
    Date=('Date', 'first'),
    Mortality=('Mortality', 'first'),
    Death_Date=('Death_Date', 'first')
)

# Compute days_until_death: calendar days from current Date to Death_Date
# Only meaningful for deceased patients (Mortality == 1)
day_df['days_until_death'] = np.nan
mask_deceased = day_df['Mortality'] == 1

day_df.loc[mask_deceased, 'days_until_death'] = day_df.loc[mask_deceased].apply(
    lambda row: (
        pd.Timestamp(row['Death_Date']) - pd.Timestamp(row['Date'])
    ).days if pd.notna(row['Death_Date']) and pd.notna(row['Date']) else np.nan,
    axis=1
)

print(f"Day-level records        : {len(day_df):,}")
print(f"Unique (PID, Visit) pairs: {day_df.groupby(['PID', 'Visit_no']).ngroups:,}")
print(f"\ndays_until_death (deceased only):")
print(day_df.loc[mask_deceased, 'days_until_death'].describe())

## 1-Day Rolling Prediction Label

| Condition | Label |
|---|---|
| Survivor (Mortality == 0) | 0 |
| `days_until_death >= 3` | 0 (safe day) |
| `days_until_death == 2` | **1** (prediction target) |
| `days_until_death == 1` | NaN — **excluded** (gap day) |
| `days_until_death == 0` | NaN — **excluded** (death day) |

In [ ]:
def assign_label_1day(row):
    if row['Mortality'] == 0:
        return 0
    dud = row['days_until_death']
    if pd.isna(dud):
        return np.nan
    if dud == 0 or dud == 1:
        return np.nan   # death day and gap day — excluded
    elif dud == 2:
        return 1        # prediction target
    else:
        return 0        # safe (3+ days before death)

day_df['label_1day'] = day_df.apply(assign_label_1day, axis=1)

labels_1day = day_df[day_df['label_1day'].notna()].copy()
labels_1day['label_1day'] = labels_1day['label_1day'].astype(int)

print('=' * 50)
print('1-Day Rolling Label Distribution')
print('=' * 50)
print(labels_1day['label_1day'].value_counts().sort_index())
print(f"\nTotal days retained : {len(labels_1day):,}")
print(f"Total days excluded : {len(day_df) - len(labels_1day):,} (death day + gap day)")
print(f"\nBreakdown by Mortality x Label:")
print(labels_1day.groupby(['Mortality', 'label_1day']).size().unstack(fill_value=0))

## 2-Day Rolling Prediction Label

| Condition | Label |
|---|---|
| Survivor (Mortality == 0) | 0 |
| `days_until_death >= 4` | 0 (safe day) |
| `days_until_death == 2` or `== 3` | **1** (within 2-day prediction window) |
| `days_until_death == 1` | NaN — **excluded** (gap day) |
| `days_until_death == 0` | NaN — **excluded** (death day) |

In [ ]:
def assign_label_2day(row):
    if row['Mortality'] == 0:
        return 0
    dud = row['days_until_death']
    if pd.isna(dud):
        return np.nan
    if dud == 0 or dud == 1:
        return np.nan        # death day and gap day — excluded
    elif dud == 2 or dud == 3:
        return 1             # within 2-day prediction window
    else:
        return 0             # safe (4+ days before death)

day_df['label_2day'] = day_df.apply(assign_label_2day, axis=1)

labels_2day = day_df[day_df['label_2day'].notna()].copy()
labels_2day['label_2day'] = labels_2day['label_2day'].astype(int)

print('=' * 50)
print('2-Day Rolling Label Distribution')
print('=' * 50)
print(labels_2day['label_2day'].value_counts().sort_index())
print(f"\nTotal days retained : {len(labels_2day):,}")
print(f"Total days excluded : {len(day_df) - len(labels_2day):,} (death day + gap day)")
print(f"\nBreakdown by Mortality x Label:")
print(labels_2day.groupby(['Mortality', 'label_2day']).size().unstack(fill_value=0))

## 7-Day Non-Rolling Prediction Label

This is **not rolling**. For each `(PID, Visit_no)`:
- Only **Days 1–5** are kept as input days
- Day 6 = implicit gap (not labeled)
- Day 7 = prediction target
- **label = 1** if the calendar date of Day 7 equals the patient's `Death_Date`
- **label = 0** otherwise (survivor, visit < 7 days, or death not on Day 7)

In [ ]:
def assign_7day_labels(group):
    """
    Non-rolling 7-day label per (PID, Visit_no).
    Returns a Series of labels aligned to group's index.
    Only Days 1-5 receive a label; all other days get NaN.
    """
    mortality_val = group['Mortality'].iloc[0]
    death_date_raw = group['Death_Date'].iloc[0]

    mask_1_5 = group['Day'].between(1, 5)
    result = pd.Series(np.nan, index=group.index)

    if mortality_val == 0:
        # Survivor: all days 1-5 are label 0
        result[mask_1_5] = 0
        return result

    # Deceased: check if Day 7 exists and its date matches Death_Date
    day7_rows = group[group['Day'] == 7]

    if day7_rows.empty:
        # Visit has fewer than 7 days — cannot make prediction
        result[mask_1_5] = 0
        return result

    date_day7 = pd.to_datetime(day7_rows['Date'].iloc[0]).date()
    death_date_obj = pd.to_datetime(death_date_raw).date() if pd.notna(death_date_raw) else None

    if death_date_obj is not None and date_day7 == death_date_obj:
        result[mask_1_5] = 1   # Death falls on Day 7
    else:
        result[mask_1_5] = 0   # Death not on Day 7

    return result


day_df['label_7day'] = day_df.groupby(
    ['PID', 'Visit_no'], group_keys=False
).apply(assign_7day_labels)

labels_7day = day_df[day_df['label_7day'].notna()].copy()
labels_7day['label_7day'] = labels_7day['label_7day'].astype(int)

pos_visits = (
    labels_7day[labels_7day['label_7day'] == 1]
    [['PID', 'Visit_no']].drop_duplicates()
)

print('=' * 55)
print('7-Day Non-Rolling Label Distribution (Days 1-5 only)')
print('=' * 55)
print(labels_7day['label_7day'].value_counts().sort_index())
print(f"\nTotal days retained (Days 1-5) : {len(labels_7day):,}")
print(f"Days excluded (Day 6+)         : {len(day_df) - len(labels_7day):,}")
print(f"Visits with label=1            : {len(pos_visits):,}")
print(f"\nBreakdown by Mortality x Label:")
print(labels_7day.groupby(['Mortality', 'label_7day']).size().unstack(fill_value=0))

In [ ]:
# Save all three label DataFrames
output_dir = Path(base_dir) / 'Processed' / 'Labels'
output_dir.mkdir(parents=True, exist_ok=True)

base_cols = ['PID', 'Visit_no', 'Day', 'Date', 'Mortality', 'Death_Date', 'days_until_death']

labels_1day[base_cols + ['label_1day']].to_csv(output_dir / 'labels_1day.csv', index=False)
print(f"Saved labels_1day.csv : {len(labels_1day):,} rows")

labels_2day[base_cols + ['label_2day']].to_csv(output_dir / 'labels_2day.csv', index=False)
print(f"Saved labels_2day.csv : {len(labels_2day):,} rows")

labels_7day[base_cols + ['label_7day']].to_csv(output_dir / 'labels_7day.csv', index=False)
print(f"Saved labels_7day.csv : {len(labels_7day):,} rows")

print(f"\nAll labels saved to: {output_dir}")

In [ ]:
# Summary comparison across all three label schemes
print('=' * 65)
print('SUMMARY: Label Comparison Across Prediction Schemes')
print('=' * 65)
print(f"{'Scheme':<12} {'Total days':>12} {'label=0':>10} {'label=1':>10} {'% positive':>12}")
print('-' * 65)

for name, df_lab, col in [
    ('1-Day',  labels_1day, 'label_1day'),
    ('2-Day',  labels_2day, 'label_2day'),
    ('7-Day',  labels_7day, 'label_7day'),
]:
    n0    = (df_lab[col] == 0).sum()
    n1    = (df_lab[col] == 1).sum()
    total = len(df_lab)
    pct   = 100.0 * n1 / total if total > 0 else 0
    print(f"{name:<12} {total:>12,} {n0:>10,} {n1:>10,} {pct:>11.2f}%")

print('=' * 65)